# Vajrastra Colab Worker

One-tap worker for the Vajrastra autonomous engineer. Run the cells in order:

1. **Setup** -- reads `GITHUB_TOKEN` and `GEMINI_API_KEY` from Colab Secrets, clones the repository and installs `z3-solver sympy hypothesis gradio`.
2. **Console** -- launches a `share=True` Gradio UI with Upload & Hyper-Fix, Code Architect, ML Scientist and Memory & Self-Evolve tabs.
3. **Worker** -- headless loop over `python vajrastra.py worker`, which polls `workspace/inbox/` and commits a checkpoint after every task.

Add both secrets under the key icon in the left sidebar before running cell 1.

In [ ]:
# Cell 1 -- one-tap setup for Google Colab (secrets, clone, engines)
from google.colab import userdata
import os, subprocess, sys

def _secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")

GITHUB_TOKEN = _secret("GITHUB_TOKEN")
GEMINI_API_KEY = _secret("GEMINI_API_KEY")
if not GITHUB_TOKEN:
    raise SystemExit("Add GITHUB_TOKEN under Colab Secrets (key icon) and allow notebook access.")

os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
os.environ["VAJRASTRA_GITHUB_TOKEN"] = GITHUB_TOKEN
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

REPO = "teamhecked-Ayu/Vajrastra"
WORKDIR = "/content/Vajrastra"
if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", WORKDIR], check=True)
os.chdir(WORKDIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "z3-solver", "sympy", "hypothesis", "gradio"],
    check=True,
)
print("repo      :", os.getcwd())
print("token     :", "loaded" if GITHUB_TOKEN else "missing")
print("git sha   :", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# Cell 2 -- Android-friendly Gradio Web UI (share=True, 4 tabs)
import json, os, shutil, sys, zipfile
from pathlib import Path

sys.path.insert(0, os.getcwd())
import gradio as gr

from core.memory_engine import MemoryEngine
from core.model_router import ModelRouter
from core.security_guard import SecurityGuard
from engines.code_architect import ArchitectError, CodeArchitect
from engines.hyper_debugger import HyperDebugger
from engines.ml_scientist import MLScientist
from engines.self_evolver import SelfEvolver

ROOT = Path(os.getcwd())
UPLOADS = ROOT / "memory_bank" / "uploads"
UPLOADS.mkdir(parents=True, exist_ok=True)
guard = SecurityGuard(ROOT)
memory = MemoryEngine(ROOT)
router = ModelRouter()


def _finding_lines(analysis, limit=20):
    lines = [
        f"- files: **{len(analysis.files)}** | findings: **{len(analysis.findings)}** "
        f"(blocking **{analysis.blocking_count}**)"
    ]
    for finding in analysis.findings[:limit]:
        lines.append(f"- `{finding.kind}` {finding.severity}: {finding.message} ({finding.path}:{finding.line})")
    return lines


def _materialise(upload):
    """Copy the browser upload into the workspace and unzip archives."""
    source = Path(getattr(upload, "name", upload))
    target = UPLOADS / source.name
    shutil.copy2(source, target)
    if target.suffix.lower() == ".zip":
        unpacked = UPLOADS / (target.stem + "-unpacked")
        unpacked.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target) as archive:
            archive.extractall(unpacked)
        return unpacked
    return target


def hyperfix(upload, repair, apply_patch):
    if not upload:
        return "Upload a file or zip first.", None
    target = _materialise(upload)
    debugger = HyperDebugger(ROOT, guard=guard, memory=memory, router=router)
    try:
        analysis = debugger.analyze(target, use_llm=True)
        lines = ["## Hyper-fix report", f"Target: `{target.relative_to(ROOT)}`", ""] + _finding_lines(analysis)
        download = None
        if repair:
            patch = debugger.repair(target, analysis, apply=bool(apply_patch))
            lines += ["", "## Verified patch", f"- verified: **{patch.verified}**", f"- applied: **{patch.applied}**",
                      f"- root cause: {patch.root_cause or 'n/a'}", f"- status: {patch.message}"]
            if patch.diff:
                lines += ["", "```diff", patch.diff.rstrip()[:4000], "```"]
            if patch.verified and apply_patch and target.is_file():
                download = str(target)
        return "\n".join(lines), download
    finally:
        debugger.close()


def architect(prompt, language, rounds):
    if not prompt.strip():
        return "Write a prompt first.", None
    if not router.available:
        return "Set GEMINI_API_KEY in Colab Secrets to use the architect.", None
    out_dir = ROOT / "memory_bank" / "architect"
    engine = CodeArchitect(ROOT, router=router, guard=guard, memory=memory)
    try:
        result = engine.build(prompt, language=language, rounds=int(rounds), out=str(out_dir))
        lines = ["## Architect result", f"- verified: **{result.verified}**", f"- rounds: {result.rounds}",
                 f"- lesson: #{result.lesson_id}", f"- summary: {result.summary[:400]}"]
        for path in result.written:
            lines.append(f"- wrote `{path}`")
        for failure in result.failures:
            lines.append(f"- failure: {failure[:200]}")
        download = None
        if result.written:
            bundle = ROOT / "memory_bank" / "architect" / "bundle.zip"
            with zipfile.ZipFile(bundle, "w") as archive:
                for name in list(result.written) + list(result.tests):
                    candidate = Path(name)
                    if candidate.is_file():
                        archive.write(candidate, candidate.name)
            download = str(bundle)
        return "\n".join(lines), download
    except ArchitectError as exc:
        return f"Architect error: {exc}", None
    finally:
        engine.close()


def scientist(topic, seeds, budget):
    if not topic.strip():
        return "Write a research topic first.", None
    engine = MLScientist(ROOT, router=router, guard=guard, memory=memory, seeds=int(seeds), budget=int(budget))
    run = engine.run(topic)
    lines = ["## MCTS research run", f"- run: `{run.run_id}`", f"- seeds: {run.seeds} | budget: {run.budget}",
             f"- baseline mean: {run.baseline.mean}", f"- gate: **{run.gate.status}**", f"- reason: {run.gate.reason}"]
    if run.best is not None:
        lines.append(f"- best node: `{run.best.node_id}` {json.dumps(run.best.params, sort_keys=True)}")
        lines.append(f"- best mean: {run.best.experiment.mean}")
    for review in run.reviews:
        if review.skipped:
            lines.append(f"- {review.persona}: skipped ({review.error})")
        else:
            lines.append(f"- {review.persona}: {review.verdict[:200]}")
    lines.append("")
    lines.append(run.metrics_tex_path.read_text(encoding="utf-8")[:2500])
    download = str(run.metrics_tex_path) if run.metrics_tex_path.is_file() else None
    return "\n".join(lines), download


def memory_panel():
    stats = memory.stats()
    unresolved = memory.unresolved(limit=10)
    lines = ["## Ledger", f"- lessons: **{stats['lessons']}** ({stats['verified_patches']} verified, {stats['unsolved']} open)",
             f"- hits: {stats['total_hits']} | skills: {stats['skills']} | fts5: {stats['fts_enabled']}"]
    for lesson in unresolved:
        lines.append(f"- open #{lesson.id}: {lesson.error_signature.splitlines()[0][:120]}")
    return "\n".join(lines)


def evolve_now(push):
    engine = SelfEvolver(ROOT, guard=guard, memory=memory, router=router)
    report = engine.review(push=bool(push))
    lines = ["## Evolution pass", f"- integrity: **{report.integrity_status}**",
             f"- branch: `{report.staging_branch}`", f"- findings: {len(report.findings)}",
             f"- accepted: {len(report.accepted)} | rejected: {len(report.rejected)}",
             f"- pushed: {', '.join(report.pushed) or 'local only'}", f"- report: `{report.report_path.relative_to(ROOT)}`"]
    for candidate in report.candidates:
        lines.append(f"- {candidate.candidate_id} `{candidate.path}` -> {'accepted' if candidate.accepted else candidate.rejection_reason}")
    lines.append(f"- meta-critic: {report.meta_critique[:400]}")
    return "\n".join(lines)


with gr.Blocks(title="Vajrastra Console") as demo:
    gr.Markdown("# Vajrastra Console\nAutonomous debugger, architect, ML scientist and memory -- powered by the Colab runtime.")
    with gr.Tab("Upload & Hyper-Fix"):
        upload = gr.File(label="Any file or .zip", type="filepath")
        with gr.Row():
            repair = gr.Checkbox(value=True, label="Repair with Blue/Red agents")
            apply_patch = gr.Checkbox(value=True, label="Write the verified patch back")
        fix_button = gr.Button("Run hyper-fix", variant="primary")
        fix_output = gr.Markdown()
        fix_file = gr.File(label="Patched file")
        fix_button.click(hyperfix, [upload, repair, apply_patch], [fix_output, fix_file])
    with gr.Tab("Code Architect"):
        prompt = gr.Textbox(label="What should Vajrastra build?", lines=3)
        with gr.Row():
            language = gr.Dropdown(["python", "javascript", "bash", "sql"], value="python", label="Language")
            rounds = gr.Slider(1, 5, value=3, step=1, label="Red/Blue rounds")
        build_button = gr.Button("Build verified code", variant="primary")
        build_output = gr.Markdown()
        build_file = gr.File(label="Verified bundle")
        build_button.click(architect, [prompt, language, rounds], [build_output, build_file])
    with gr.Tab("ML Scientist"):
        topic = gr.Textbox(label="Research topic / hypothesis", lines=2)
        with gr.Row():
            seeds = gr.Slider(3, 5, value=3, step=1, label="Paired seeds")
            budget = gr.Slider(2, 40, value=10, step=1, label="MCTS budget")
        research_button = gr.Button("Run MCTS", variant="primary")
        research_output = gr.Markdown()
        research_file = gr.File(label="metrics.tex")
        research_button.click(scientist, [topic, seeds, budget], [research_output, research_file])
    with gr.Tab("Memory & Self-Evolve"):
        memory_button = gr.Button("Inspect lessons.sqlite")
        memory_output = gr.Markdown()
        push = gr.Checkbox(value=False, label="Push staged candidates to self-upgrade/<timestamp>")
        evolve_button = gr.Button("Run evolve", variant="primary")
        evolve_output = gr.Markdown()
        memory_button.click(memory_panel, None, memory_output)
        evolve_button.click(evolve_now, [push], evolve_output)

demo.launch(share=True)


In [ ]:
# Cell 3 -- headless background queue worker with auto-commit checkpoints
import os, subprocess, sys, time
from datetime import datetime

WORKDIR = "/content/Vajrastra"
INTERVAL = 30      # seconds between polls
POLLS = 0          # 0 = run until you stop the cell
MAX_POLLS = 0      # set to a positive number for a bounded run

os.chdir(WORKDIR)
env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")

def log(message):
    print(f"[{datetime.utcnow().strftime('%H:%M:%S')}] {message}", flush=True)

log("worker online; polling workspace/inbox/")
while MAX_POLLS <= 0 or POLLS < MAX_POLLS:
    result = subprocess.run(
        [sys.executable, "vajrastra.py", "worker", "--once", "--max-jobs", "3"],
        capture_output=True, text=True, env=env,
    )
    output = (result.stdout or "").strip() or (result.stderr or "").strip()
    log(output if output else "no output")
    if result.returncode not in (0, 1):
        log(f"worker exited {result.returncode}")
    POLLS += 1
    time.sleep(INTERVAL)
log("worker stopped")
